# RAG-based Scientific Paper Question Answering Assistant

This notebook connects the full project pipeline for a GPU virtual machine workflow.

Core principle:

1. Run the complete pipeline in mock mode first.
2. After data loading, chunking, indexing, retrieval, evaluation CSV output, and UI logic work, switch to real GPU inference.

Note: this project does not fine-tune Qwen or Mistral. The VM workload is dataset preparation, embedding/index construction, retrieval, LLM inference, and evaluation.

## 0. Runtime Plan

Recommended execution order:

1. Configure notebook flags.
2. Install dependencies if needed.
3. Load a small QASPER sample.
4. Chunk paper text.
5. Build the FAISS index with `BAAI/bge-m3`.
6. Test retrieval.
7. Test answer generation in mock mode.
8. Run mock evaluation end to end.
9. Set `MOCK_MODE = False` and `RUN_REAL_MODE = True` on a GPU VM.
10. Run real Qwen/Mistral evaluation.
11. Launch Streamlit for the demo.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

# If the notebook is opened from notebooks/, move to the repository root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(f"Could not find project root from {Path.cwd()}")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

## 1. Configuration

Leave `MOCK_MODE = True` for the first full pipeline run. On the GPU VM, after the mock run succeeds, set `MOCK_MODE = False` and `RUN_REAL_MODE = True`.

In [ ]:
RUN_INSTALL = False

# First pass should stay True. Real 7B inference should only run on the GPU VM.
MOCK_MODE = True
RUN_REAL_MODE = False

NUM_PAPERS = 10
MAX_QUESTIONS = 30
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 5
MAX_SAMPLES = 30

QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
EMBEDDING_MODEL = "BAAI/bge-m3"

# Keep model/cache files outside Git-tracked outputs and the inode-limited /root filesystem.
os.environ.setdefault("DISABLE_SAFETENSORS_CONVERSION", "true")
os.environ.setdefault("HF_HOME", "/tmp/rag-paper-qa-hf-cache")
os.environ.setdefault("HF_HUB_CACHE", "/tmp/rag-paper-qa-hf-cache/hub")
os.environ.setdefault("HF_DATASETS_CACHE", "/tmp/rag-paper-qa-hf-cache/datasets")
os.environ.setdefault("SENTENCE_TRANSFORMERS_HOME", "/tmp/rag-paper-qa-sentence-transformers")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/rag-paper-qa-xdg-cache")
os.environ.setdefault("TMPDIR", "/tmp")

print("MOCK_MODE:", MOCK_MODE)
print("RUN_REAL_MODE:", RUN_REAL_MODE)
print("HF_HOME:", os.environ.get("HF_HOME"))
print("SENTENCE_TRANSFORMERS_HOME:", os.environ.get("SENTENCE_TRANSFORMERS_HOME"))

## 2. Install Dependencies

Set `RUN_INSTALL = True` above if the VM environment has not installed the project dependencies yet. For GPU use, install a CUDA-compatible PyTorch build for the VM image.

In [ ]:
if RUN_INSTALL:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
        check=True,
    )
else:
    print("Skipping install. Set RUN_INSTALL=True if dependencies are missing.")

## 3. Helper Function for Project Scripts

In [ ]:
def run_python(args):
    """Run a project Python script with the active notebook interpreter."""
    command = [sys.executable, *args]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def preview_json(path, limit=1):
    data = load_json(path)
    print(f"{path}: {len(data) if isinstance(data, list) else 'non-list'} records")
    if isinstance(data, list) and data:
        print(json.dumps(data[:limit], ensure_ascii=False, indent=2)[:3000])
    return data

## 4. Load QASPER Sample

This step downloads `allenai/qasper`, samples papers, extracts QA pairs, and writes:

- `data/qasper_sample.json`
- `data/qa_samples.json`

In [ ]:
run_python([
    "src/load_data.py",
    "--num_papers", str(NUM_PAPERS),
    "--max_questions", str(MAX_QUESTIONS),
])

papers = preview_json("data/qasper_sample.json", limit=1)
qa_samples = preview_json("data/qa_samples.json", limit=2)

## 5. Chunk Paper Text

This step splits each paper into overlapping word chunks and writes `data/chunks.json`.

In [ ]:
run_python([
    "src/chunk_text.py",
    "--chunk_size", str(CHUNK_SIZE),
    "--chunk_overlap", str(CHUNK_OVERLAP),
])

chunks = preview_json("data/chunks.json", limit=2)

## 6. Build FAISS Index

This step loads `BAAI/bge-m3`, embeds all chunks, normalizes embeddings, builds a FAISS index, and writes:

- `data/faiss.index`
- `data/chunk_metadata.json`

This is usually safe on CPU for the small sample, but it can still download the embedding model.

In [ ]:
run_python([
    "src/build_index.py",
    "--model_name", EMBEDDING_MODEL,
    "--batch_size", "8",
])

print("FAISS index exists:", Path("data/faiss.index").exists())
metadata = preview_json("data/chunk_metadata.json", limit=2)

## 7. Test Retrieval

Retrieve top-k evidence for one QA sample, limited to the selected paper.

In [ ]:
from src.retrieve import Retriever

sample = qa_samples[0]
question = sample["question"]
paper_id = sample["paper_id"]

retriever = Retriever()
retrieved_chunks = retriever.retrieve(question, paper_id=paper_id, top_k=TOP_K)

print("Question:", question)
print("Gold answer:", sample.get("gold_answer"))
print(json.dumps(retrieved_chunks, ensure_ascii=False, indent=2)[:4000])

## 8. Test Answer Generation

With `MOCK_MODE = True`, this cell does not load Qwen or Mistral. After the full mock pipeline succeeds on the VM, set `MOCK_MODE = False` to run real generation.

In [ ]:
from src.answer import generate_answer

direct_answer = generate_answer(
    model_name=QWEN_MODEL,
    question=question,
    mode="direct",
    mock=MOCK_MODE,
)

rag_answer = generate_answer(
    model_name=QWEN_MODEL,
    question=question,
    evidence=retrieved_chunks,
    mode="rag",
    mock=MOCK_MODE,
)

print("Direct answer:\n", direct_answer)
print("\nRAG answer:\n", rag_answer)

## 9. Run End-to-End Mock Evaluation

This should be the first full successful run. It verifies that retrieval, answer-generation calls, checkpointing, and CSV output are connected correctly.

In [ ]:
eval_args = [
    "src/evaluate.py",
    "--max_samples", str(MAX_SAMPLES),
    "--top_k", str(TOP_K),
]
if MOCK_MODE:
    eval_args.append("--mock")

run_python(eval_args)

results = pd.read_csv("results/comparison_results.csv")
display(results.head())
print("Rows:", len(results))
print("Evidence Hit Rate:", results["evidence_hit"].mean() if len(results) else None)

## 10. Real GPU Evaluation

Only run this after the mock evaluation has succeeded. Requirements:

- GPU VM with enough VRAM or working `device_map='auto'` offload.
- CUDA-compatible PyTorch installation.
- Hugging Face authentication if the selected model requires it.
- Accepted Mistral model terms on Hugging Face if required.

Set `MOCK_MODE = False` and `RUN_REAL_MODE = True` in the configuration cell, then rerun from this cell or rerun the notebook from the start.

In [ ]:
if RUN_REAL_MODE:
    run_python([
        "src/evaluate.py",
        "--max_samples", str(MAX_SAMPLES),
        "--top_k", str(TOP_K),
    ])
    real_results = pd.read_csv("results/comparison_results.csv")
    display(real_results.head())
else:
    print("Skipping real evaluation. Set RUN_REAL_MODE=True on the GPU VM.")

## 11. Manual Scoring Template

After real evaluation, fill these CSV columns manually:

- `direct_qwen_score`
- `rag_qwen_score`
- `direct_mistral_score`
- `rag_mistral_score`

Scoring rubric:

- `0`: wrong
- `1`: partially correct
- `2`: correct

In [ ]:
results_path = Path("results/comparison_results.csv")
if results_path.exists():
    df = pd.read_csv(results_path)
    score_cols = [
        "direct_qwen_score",
        "rag_qwen_score",
        "direct_mistral_score",
        "rag_mistral_score",
    ]
    display(df[["sample_id", "question", "gold_answer", *score_cols]].head())
else:
    print("No results CSV found yet.")

## 12. Launch Streamlit Demo

Run this command in a VM terminal. It stays running, so it is usually better to start it outside the notebook.

In [ ]:
print("streamlit run src/app.py --server.address 0.0.0.0 --server.port 8501")

## 13. Submission Checklist

Expected final files:

- `README.md`
- `requirements.txt`
- `src/` implementation code
- `results/comparison_results.csv`
- `summary_slides.pptx`
- `demo.mp4`

Do not commit model weights, Hugging Face caches, or generated FAISS indexes.